# Reverse-Engineering the Negation Circuit in GPT-2 Small

**Research question:** Which attention heads and MLP layers in GPT-2 small causally mediate negation's effect on sentiment prediction, and does this circuit generalize across sentence structure and negation word?

**Run this in Google Colab** (Runtime -> Change runtime type -> GPU, T4 is enough).
This notebook needs internet access to Hugging Face, which this dev sandbox doesn't have -
that's why we're running it here instead.

Pipeline: baseline check -> activation patching -> path patching -> direct logit attribution
-> ablation -> stress tests -> visualization export.

## 0. Setup

In [ ]:
!pip install -q transformer_lens

import json
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer
from functools import partial

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = HookedTransformer.from_pretrained("gpt2")
model = model.to(device)
n_layers, n_heads = model.cfg.n_layers, model.cfg.n_heads
print(f"GPT-2 small: {n_layers} layers x {n_heads} heads")

## 1. Load dataset & verify tokenization

Upload `data/train.jsonl` and `data/test.jsonl` from this repo (or clone the repo in Colab).
We first confirm every chosen adjective is a single token - required for clean logit-diff
measurement. Any pair that isn't single-token on both sides is dropped.

In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f]

train_rows = load_jsonl("data/train.jsonl")
test_rows = load_jsonl("data/test.jsonl")

def is_single_token(word):
    return len(model.to_tokens(word, prepend_bos=False)[0]) == 1

def filter_single_token(rows):
    kept = [r for r in rows if is_single_token(r["positive_adj"]) and is_single_token(r["negative_adj"])]
    print(f"Kept {len(kept)}/{len(rows)} (dropped multi-token adjective pairs)")
    return kept

train_rows = filter_single_token(train_rows)
test_rows = filter_single_token(test_rows)

print("\nExample:")
print(train_rows[0])

## 2. Baseline: does GPT-2 actually shift its prediction under negation?

This is the sanity check before any interpretability work. If the behavior isn't real
and robust, there's no circuit worth finding.

In [ ]:
def logit_diff_for_prompt(prompt, pos_adj, neg_adj):
    tokens = model.to_tokens(prompt)
    logits = model(tokens)
    final_logits = logits[0, -1]
    pos_id = model.to_single_token(pos_adj)
    neg_id = model.to_single_token(neg_adj)
    return (final_logits[pos_id] - final_logits[neg_id]).item()

affirmative_diffs, negated_diffs = [], []
for r in train_rows[:150]:  # subsample for speed; full set used later for the headline number
    affirmative_diffs.append(logit_diff_for_prompt(r["affirmative_prompt"], r["positive_adj"], r["negative_adj"]))
    negated_diffs.append(logit_diff_for_prompt(r["negated_prompt"], r["positive_adj"], r["negative_adj"]))

print(f"Affirmative mean logit_diff (pos - neg adj): {np.mean(affirmative_diffs):.3f}")
print(f"Negated     mean logit_diff (pos - neg adj): {np.mean(negated_diffs):.3f}")
print(f"Shift caused by negation: {np.mean(affirmative_diffs) - np.mean(negated_diffs):.3f}")
print("\nIf affirmative is clearly positive and negated is clearly smaller/negative,")
print("the behavior is real and we proceed to find the circuit that causes it.")

## 3. Activation patching: which layer/head causes the shift?

Method: run the NEGATED prompt ("corrupted"), but patch in the activation from the
AFFIRMATIVE prompt ("clean") at one component at a time. If patching a component
restores the affirmative-like logit_diff, that component is causally implicated.

We patch every (layer, head) attention output and every layer's MLP output at the
final token position, and record how much each patch moves the logit_diff back
toward the clean (affirmative) value.

In [ ]:
def get_logit_diff_from_logits(logits, pos_id, neg_id):
    return logits[0, -1, pos_id] - logits[0, -1, neg_id]

def run_patching_for_pair(row, n_layers=n_layers, n_heads=n_heads):
    clean_tokens = model.to_tokens(row["affirmative_prompt"])
    corrupted_tokens = model.to_tokens(row["negated_prompt"])
    pos_id = model.to_single_token(row["positive_adj"])
    neg_id = model.to_single_token(row["negative_adj"])

    # skip pairs whose tokenized length differs - patching needs matched positions
    if clean_tokens.shape[1] != corrupted_tokens.shape[1]:
        return None

    _, clean_cache = model.run_with_cache(clean_tokens)
    corrupted_logits = model(corrupted_tokens)
    clean_logits = model(clean_tokens)

    clean_diff = get_logit_diff_from_logits(clean_logits, pos_id, neg_id).item()
    corrupted_diff = get_logit_diff_from_logits(corrupted_logits, pos_id, neg_id).item()

    head_results = torch.zeros(n_layers, n_heads)
    mlp_results = torch.zeros(n_layers)

    def patch_head_hook(activation, hook, layer, head):
        activation[:, -1, head, :] = clean_cache[hook.name][:, -1, head, :]
        return activation

    def patch_mlp_hook(activation, hook, layer):
        activation[:, -1, :] = clean_cache[hook.name][:, -1, :]
        return activation

    for layer in range(n_layers):
        for head in range(n_heads):
            hook_fn = partial(patch_head_hook, layer=layer, head=head)
            patched_logits = model.run_with_hooks(
                corrupted_tokens,
                fwd_hooks=[(f"blocks.{layer}.attn.hook_z", hook_fn)]
            )
            patched_diff = get_logit_diff_from_logits(patched_logits, pos_id, neg_id).item()
            # normalize: 0 = no recovery, 1 = full recovery to clean
            denom = (clean_diff - corrupted_diff)
            head_results[layer, head] = (patched_diff - corrupted_diff) / denom if abs(denom) > 1e-6 else 0.0

        hook_fn = partial(patch_mlp_hook, layer=layer)
        patched_logits = model.run_with_hooks(
            corrupted_tokens,
            fwd_hooks=[(f"blocks.{layer}.hook_mlp_out", hook_fn)]
        )
        patched_diff = get_logit_diff_from_logits(patched_logits, pos_id, neg_id).item()
        denom = (clean_diff - corrupted_diff)
        mlp_results[layer] = (patched_diff - corrupted_diff) / denom if abs(denom) > 1e-6 else 0.0

    return head_results, mlp_results

# Average patching results over a sample of prompt pairs for a stable signal
N_SAMPLE = 20
all_head_results, all_mlp_results = [], []
used = 0
for r in train_rows:
    out = run_patching_for_pair(r)
    if out is None:
        continue
    all_head_results.append(out[0])
    all_mlp_results.append(out[1])
    used += 1
    if used >= N_SAMPLE:
        break

mean_head_results = torch.stack(all_head_results).mean(0)
mean_mlp_results = torch.stack(all_mlp_results).mean(0)
print(f"Averaged over {used} prompt pairs")

In [ ]:
# Visualize: heatmap of which heads matter most
fig, axes = plt.subplots(1, 2, figsize=(14, 5), gridspec_kw={"width_ratios": [3, 1]})

im = axes[0].imshow(mean_head_results, cmap="RdBu_r", vmin=-0.3, vmax=0.3, aspect="auto")
axes[0].set_xlabel("Head")
axes[0].set_ylabel("Layer")
axes[0].set_title("Activation patching: recovery per attention head")
plt.colorbar(im, ax=axes[0])

axes[1].barh(range(n_layers), mean_mlp_results.numpy())
axes[1].set_title("MLP recovery per layer")
axes[1].set_xlabel("Recovery")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig("figures/patching_heatmap.png", dpi=150)
plt.show()

# Print top-5 heads by |recovery|
flat = mean_head_results.flatten()
top_idx = flat.abs().argsort(descending=True)[:5]
print("Top 5 heads by causal effect:")
for idx in top_idx:
    layer, head = idx.item() // n_heads, idx.item() % n_heads
    print(f"  L{layer}H{head}: recovery = {mean_head_results[layer, head]:.3f}")

## 4. Direct logit attribution

Independent check: instead of patching, decompose the final logit_diff additively
into each component's direct contribution (via the residual stream). Components that
matter in BOTH activation patching and direct logit attribution are the most trustworthy
findings - two independent methods agreeing is what makes this robust, not a coincidence.

In [ ]:
def direct_logit_attribution(row):
    tokens = model.to_tokens(row["negated_prompt"])
    pos_id = model.to_single_token(row["positive_adj"])
    neg_id = model.to_single_token(row["negative_adj"])
    logit_diff_direction = model.W_U[:, pos_id] - model.W_U[:, neg_id]

    _, cache = model.run_with_cache(tokens)
    per_head_dla = cache.stack_head_results(layer=-1, return_labels=False)[:, 0, -1, :]  # (n_layer*n_head, d_model)
    per_head_dla = per_head_dla @ logit_diff_direction
    return per_head_dla.reshape(n_layers, n_heads)

dla_results = torch.stack([direct_logit_attribution(r) for r in train_rows[:20]]).mean(0)

plt.figure(figsize=(8, 5))
plt.imshow(dla_results, cmap="RdBu_r", aspect="auto")
plt.xlabel("Head"); plt.ylabel("Layer")
plt.title("Direct logit attribution per head")
plt.colorbar()
plt.savefig("figures/dla_heatmap.png", dpi=150)
plt.show()

flat = dla_results.flatten()
top_idx = flat.abs().argsort(descending=True)[:5]
print("Top 5 heads by direct logit attribution:")
for idx in top_idx:
    layer, head = idx.item() // n_heads, idx.item() % n_heads
    print(f"  L{layer}H{head}: DLA = {dla_results[layer, head]:.3f}")

## 5. Ablation study

Confirm the suspect heads (intersection of top patching + top DLA heads) actually matter
by zero-ablating them and checking that:
  (a) the negation effect specifically breaks, and
  (b) general language modeling (loss on unrelated text) is NOT badly damaged -
      otherwise the head is just generally important, not negation-specific.

In [ ]:
# Fill in after inspecting the top heads from cells above
SUSPECT_HEADS = []  # e.g. [(9, 6), (10, 1)]

def ablate_heads_hook(activation, hook, heads_this_layer):
    for head in heads_this_layer:
        activation[:, -1, head, :] = 0.0
    return activation

def run_with_ablation(prompt, pos_adj, neg_adj, heads):
    tokens = model.to_tokens(prompt)
    pos_id = model.to_single_token(pos_adj)
    neg_id = model.to_single_token(neg_adj)
    by_layer = {}
    for l, h in heads:
        by_layer.setdefault(l, []).append(h)
    hooks = [(f"blocks.{l}.attn.hook_z", partial(ablate_heads_hook, heads_this_layer=hs))
             for l, hs in by_layer.items()]
    logits = model.run_with_hooks(tokens, fwd_hooks=hooks)
    return get_logit_diff_from_logits(logits, pos_id, neg_id).item()

if SUSPECT_HEADS:
    before, after = [], []
    for r in test_rows[:50]:
        before.append(logit_diff_for_prompt(r["negated_prompt"], r["positive_adj"], r["negative_adj"]))
        after.append(run_with_ablation(r["negated_prompt"], r["positive_adj"], r["negative_adj"], SUSPECT_HEADS))
    print(f"Mean logit_diff before ablation: {np.mean(before):.3f}")
    print(f"Mean logit_diff after ablating {SUSPECT_HEADS}: {np.mean(after):.3f}")
else:
    print("Set SUSPECT_HEADS from the top results in cells 3-4 above, then re-run this cell.")

## 6. Stress tests: does the circuit generalize?

Run the SAME suspect heads' ablation/patching effect against:
  - the held-out **test** set (unseen subjects/objects/templates)
  - a different negation word ("never" instead of "did not") - add a few by hand
  - a different sentence template entirely

A circuit that only works on the exact training template is a curve-fit, not a finding.

In [ ]:
hand_written_stress_tests = [
    {"affirmative_prompt": "He always says the coffee here is really",
     "negated_prompt": "He never says the coffee here is really",
     "positive_adj": " good", "negative_adj": " bad"},
    {"affirmative_prompt": "The teacher believes the essay was quite",
     "negated_prompt": "The teacher does not believe the essay was quite",
     "positive_adj": " great", "negative_adj": " terrible"},
]

print("Held-out test set (same templates, unseen vocab):")
test_before = [logit_diff_for_prompt(r["negated_prompt"], r["positive_adj"], r["negative_adj"]) for r in test_rows[:50]]
print(f"  mean logit_diff on negated test prompts: {np.mean(test_before):.3f}")

print("\nHand-written stress tests (new template / new negation word):")
for r in hand_written_stress_tests:
    d = logit_diff_for_prompt(r["negated_prompt"], r["positive_adj"], r["negative_adj"])
    print(f"  '{r['negated_prompt']}' -> logit_diff = {d:.3f}")

## 7. Export data for the interactive visualization

Saves per-head patching + DLA scores and a sample of attention patterns for the
top suspect head, so the website widget can render them without re-running the model.

In [ ]:
export = {
    "n_layers": n_layers,
    "n_heads": n_heads,
    "patching_scores": mean_head_results.tolist(),
    "mlp_scores": mean_mlp_results.tolist(),
    "dla_scores": dla_results.tolist(),
    "suspect_heads": SUSPECT_HEADS,
}
with open("results/circuit_export.json", "w") as f:
    json.dump(export, f, indent=2)
print("Saved results/circuit_export.json")